In [22]:
from pathlib import Path
import tensorflow_model_analysis as tfma

# project paths
PROJECT_ROOT = Path("/mnt/c/Users/said_/Documents/COMP-315-Team-Project")
EVAL_DIR = PROJECT_ROOT / "tfx_pipeline_output_v2" / "Evaluator" / "evaluation" / "8"

print("evaluation folder:", EVAL_DIR)
print("files found:")
for path in sorted(EVAL_DIR.iterdir()):
    print("-", path.name)

evaluation folder: /mnt/c/Users/said_/Documents/COMP-315-Team-Project/tfx_pipeline_output_v2/Evaluator/evaluation/8
files found:
- attributions-00000-of-00001.tfrecord
- eval_config.json
- metrics-00000-of-00001.tfrecord
- plots-00000-of-00001.tfrecord
- validations.tfrecord


In [23]:
eval_result = tfma.load_eval_result(str(EVAL_DIR))
print("tfma result loaded successfully")

tfma result loaded successfully


In [24]:
tfma.view.render_slicing_metrics(eval_result)

SlicingMetricsViewer(config={'weightedExamplesColumn': 'example_count'}, data=[{'slice': 'Overall', 'metrics':…

In [25]:
tfma.view.render_slicing_metrics(eval_result, slicing_column='sex')

SlicingMetricsViewer(config={'weightedExamplesColumn': 'example_count'}, data=[{'slice': 'sex:Male', 'metrics'…

In [26]:
tfma.view.render_slicing_metrics(eval_result, slicing_column='race')

SlicingMetricsViewer(config={'weightedExamplesColumn': 'example_count'}, data=[{'slice': 'race:White', 'metric…

In [27]:
import os
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
print("protobuf workaround set")

protobuf workaround set


In [28]:
from witwidget.notebook.visualization import WitWidget, WitConfigBuilder
print("wit import ok")

wit import ok


In [29]:
from pathlib import Path
import pandas as pd
import tensorflow as tf
from witwidget.notebook.visualization import WitWidget, WitConfigBuilder

PROJECT_ROOT = Path("/mnt/c/Users/said_/Documents/COMP-315-Team-Project")
DATA_PATH = PROJECT_ROOT / "data" / "Dataset1_adult" / "source" / "adult.csv"

# find the exported savedmodel automatically
saved_model_pb = next((PROJECT_ROOT / "tfx_pipeline_output_v2" / "Trainer" / "model" / "7").rglob("saved_model.pb"))
MODEL_DIR = saved_model_pb.parent

loaded_model = tf.saved_model.load(str(MODEL_DIR))
serve_fn = loaded_model.signatures["serving_default"]

print("model dir:", MODEL_DIR)
print("data path:", DATA_PATH)

model dir: /mnt/c/Users/said_/Documents/COMP-315-Team-Project/tfx_pipeline_output_v2/Trainer/model/7/Format-Serving
data path: /mnt/c/Users/said_/Documents/COMP-315-Team-Project/data/Dataset1_adult/source/adult.csv


In [30]:
import tensorflow_transform as tft

# find the latest transform graph automatically
transform_root = PROJECT_ROOT / "tfx_pipeline_output_v2" / "Transform" / "transform_graph"
latest_transform_dir = max(
    [p for p in transform_root.iterdir() if p.is_dir()],
    key=lambda p: int(p.name)
)

tf_transform_output = tft.TFTransformOutput(str(latest_transform_dir))

# get the exact raw feature spec used by the pipeline
raw_feature_spec = tf_transform_output.raw_feature_spec().copy()

# keep target out of model inference examples
raw_feature_spec.pop("target", None)

# use a small sample so wit stays fast
df = pd.read_csv(DATA_PATH).head(50).copy()

def row_to_example(row):
    ex = tf.train.Example()

    for feature_name, feature_spec in raw_feature_spec.items():
        if feature_name not in row:
            continue

        value = row[feature_name]

        if pd.isna(value):
            continue

        # use the exact dtype expected by the saved model
        if feature_spec.dtype == tf.int64:
            ex.features.feature[feature_name].int64_list.value.append(int(value))

        elif feature_spec.dtype == tf.float32:
            ex.features.feature[feature_name].float_list.value.append(float(value))

        else:
            ex.features.feature[feature_name].bytes_list.value.append(
                str(value).encode("utf-8")
            )

    return ex

examples = [row_to_example(row) for _, row in df.iterrows()]

print("transform dir:", latest_transform_dir)
print("examples loaded:", len(examples))
print("raw feature keys:", list(raw_feature_spec.keys()))

transform dir: /mnt/c/Users/said_/Documents/COMP-315-Team-Project/tfx_pipeline_output_v2/Transform/transform_graph/6
examples loaded: 50
raw feature keys: ['age', 'capital-gain', 'capital-loss', 'education', 'education-num', 'fnlwgt', 'hours-per-week', 'marital-status', 'native-country', 'occupation', 'race', 'relationship', 'sex', 'workclass']


In [31]:
def custom_predict_fn(examples_to_infer):
    serialized = tf.constant([ex.SerializeToString() for ex in examples_to_infer])
    preds = serve_fn(examples=serialized)["outputs"].numpy().reshape(-1)

    # wit expects class probs for binary classification
    return [[1.0 - float(p), float(p)] for p in preds]

# quick sanity check
test_preds = custom_predict_fn(examples[:3])
print(test_preds)

[[0.6412726044654846, 0.3587273955345154], [0.550542801618576, 0.44945719838142395], [0.960616122931242, 0.03938387706875801]]


In [32]:
sample_df = df.head(5).copy()
sample_examples = [row_to_example(row) for _, row in sample_df.iterrows()]
sample_preds = custom_predict_fn(sample_examples)

sample_df["p_>50K"] = [p[1] for p in sample_preds]
sample_df[["age", "sex", "race", "hours-per-week", "education-num", "p_>50K"]]

,age,sex,race,hours-per-week,education-num,p_>50K
0,39,Male,White,40,13,0.358727
1,50,Male,White,13,13,0.449457
2,38,Male,White,40,9,0.039384
3,53,Male,Black,40,7,0.181513
4,28,Female,Black,40,13,0.810763


In [33]:
base = df.iloc[0].copy()

# flip sex using whatever two values already exist in your dataset
sex_values = list(df["sex"].dropna().unique())
alt_sex = [v for v in sex_values if str(v).strip() != str(base["sex"]).strip()][0]

cf_sex = base.copy()
cf_sex["sex"] = alt_sex

cf_hours = base.copy()
cf_hours["hours-per-week"] = 60

compare_df = pd.DataFrame(
    [base, cf_sex, cf_hours],
    index=["original", "sex_changed", "hours_60"]
)

compare_examples = [row_to_example(row) for _, row in compare_df.iterrows()]
compare_preds = custom_predict_fn(compare_examples)

compare_df["p_>50K"] = [p[1] for p in compare_preds]
compare_df[["sex", "race", "hours-per-week", "education-num", "p_>50K"]]

,sex,race,hours-per-week,education-num,p_>50K
original,Male,White,40,13,0.358727
sex_changed,Female,White,40,13,0.235742
hours_60,Male,White,60,13,0.436353
